In [64]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import List, Dict, Tuple, Any, Union
import torch.nn as nn
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
from datasets import load_dataset
import nltk
from tqdm import tqdm
import os
from sklearn.model_selection import train_test_split
from typing import Tuple, List, Dict
from collections import deque
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import os
import json


# Context Enrichment for RAG Embeddings

## Description
This notebook implements a context enrichment model for RAG (Retrieval-Augmented Generation) embeddings. The model uses a hybrid architecture combining bidirectional GRU and attention mechanisms to enrich chunk embeddings with contextual information from surrounding text and document structure.

## Architecture Overview
- **Hybrid Enricher**: Combines Bi-GRU and Multi-head Attention
- **Input**: Sequence of chunk embeddings
- **Output**: Context-enriched embeddings
- **Key Features**: 
  - Bidirectional contextual processing
  - Global attention mechanism
  - Hierarchical information preservation

## Model Components

In [65]:
class HybridEnricher(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, nhead=8, num_layers=2, dropout=0.1):
        super().__init__()
        # Bi-GRU pour le traitement séquentiel initial
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Attention pour capturer les relations globales
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,  # *2 car bidirectionnel
            num_heads=nhead,
            batch_first=True,
            dropout=dropout
        )
        
        # Projection finale
        self.projection = nn.Linear(hidden_dim * 2, embedding_dim)
        self.norm = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # Traitement Bi-GRU
        gru_out, _ = self.gru(x)  # gru_out shape: (batch, seq_len, hidden_dim*2)
        
        # Self-attention avec masque optionnel pour l'attention causale si nécessaire
        attn_out, _ = self.attention(gru_out, gru_out, gru_out)
        
        # Connexion résiduelle et dropout
        combined = self.dropout(gru_out + attn_out)
        
        # Projection et normalisation
        enriched = self.projection(combined)
        return self.norm(enriched)

In [66]:
# Fonction d'utilisation avec gestion du batch et du padding
def process_sequence(chunks, embedding_model, enricher, max_seq_length=None):
    device = next(enricher.parameters()).device
    
    # Convertir les chunks en embeddings
    embeddings = [embedding_model.encode(chunk) for chunk in chunks]
    seq_length = len(embeddings)
    
    # Gérer la longueur maximale si spécifiée
    if max_seq_length:
        seq_length = min(seq_length, max_seq_length)
        embeddings = embeddings[:max_seq_length]
    
    # Convertir en tensor et ajouter dimension de batch
    input_tensor = torch.tensor(embeddings, device=device).float()
    input_tensor = input_tensor.unsqueeze(0)  # (1, seq_length, embedding_dim)
    
    # Passage dans le modèle
    with torch.no_grad():
        enriched = enricher(input_tensor)
        return enriched.squeeze(0)  # Retirer la dimension de batch



In [67]:
def train_enricher(
    model, 
    data_loader, 
    num_epochs, 
    learning_rate=1e-4, 
    device='cuda'
):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch_idx, (input_embeddings, targets) in enumerate(data_loader):
            input_embeddings = input_embeddings.to(device)
            targets = targets.to(device)
            
            # Forward pass
            enriched = model(input_embeddings)
            
            # Calculer la perte (à adapter selon vos besoins)
            loss = torch.nn.functional.mse_loss(enriched, targets)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_loss = total_loss / len(data_loader)
        print(f'Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}')

In [68]:
class ContextualEmbeddingDataset(Dataset):
    def __init__(
        self,
        documents: List[Dict],
        embedding_model,
        window_size: int = 5,
        max_seq_length: int = 50,
        use_augmentation: bool = True
    ):
        self.embedding_model = embedding_model
        self.window_size = window_size
        self.use_augmentation = use_augmentation
        print(f"Initialisation du dataset avec {len(documents)} documents")
        self.samples = self._prepare_samples(documents, max_seq_length)
        print(f"Dataset initialisé avec {len(self.samples)} échantillons")
        
    def augment_chunk(self, chunk: Dict) -> List[Dict]:
        """Augmente un chunk avec différentes variations."""
        augmented = []
        
        # Version originale
        augmented.append(chunk)
        
        if not self.use_augmentation:
            return augmented
            
        # Masquage aléatoire de mots
        def apply_random_masking(text: str, mask_prob: float = 0.15) -> str:
            words = text.split()
            masked_words = [
                '[MASK]' if np.random.random() < mask_prob else word 
                for word in words
            ]
            return ' '.join(masked_words)
        
        # Troncature aléatoire
        def truncate_text(text: str, min_keep: float = 0.5) -> str:
            words = text.split()
            keep_length = int(len(words) * np.random.uniform(min_keep, 1.0))
            return ' '.join(words[:keep_length])
        
        # Ajouter versions augmentées
        masked_text = apply_random_masking(chunk['text'])
        augmented.append({**chunk, 'text': masked_text})
        
        truncated_text = truncate_text(chunk['text'])
        augmented.append({**chunk, 'text': truncated_text})
        
        return augmented

    def _prepare_samples(self, documents: List[Dict], max_seq_length: int) -> List[Tuple]:
        samples = []
        total_chunks = 0
        
        print(f"Préparation des échantillons pour {len(documents)} documents")
        
        for doc_idx, doc in enumerate(documents):
            try:
                chunks_with_meta = self._extract_chunks_with_metadata(doc)
                total_chunks += len(chunks_with_meta)
                print(f"Document {doc_idx}: {len(chunks_with_meta)} chunks extraits")
                
                if not chunks_with_meta:
                    print(f"Warning: Aucun chunk extrait pour le document {doc_idx}: {doc.get('title', 'Unknown')}")
                    continue
                
                for i in range(len(chunks_with_meta)):
                    start_idx = max(0, i - self.window_size)
                    end_idx = min(len(chunks_with_meta), i + self.window_size + 1)
                    sequence = chunks_with_meta[start_idx:end_idx]
                    
                    # Appliquer l'augmentation sur le chunk central
                    try:
                        augmented_chunks = self.augment_chunk(chunks_with_meta[i])
                        
                        for aug_chunk in augmented_chunks:
                            try:
                                target = self._create_target_embedding(aug_chunk, sequence)
                                input_embeddings = self._create_input_sequence(sequence)
                                samples.append((input_embeddings, target))
                            except Exception as e:
                                print(f"Erreur lors de la création de l'échantillon dans le document {doc_idx}, chunk {i}: {str(e)}")
                                
                    except Exception as e:
                        print(f"Erreur lors de l'augmentation dans le document {doc_idx}, chunk {i}: {str(e)}")
                
            except Exception as e:
                print(f"Erreur lors du traitement du document {doc_idx}: {str(e)}")
        
        print(f"Statistiques de préparation des données:")
        print(f"- Documents traités: {len(documents)}")
        print(f"- Total chunks extraits: {total_chunks}")
        print(f"- Échantillons créés: {len(samples)}")
        
        if len(samples) == 0:
            print("WARNING: Aucun échantillon n'a été créé!")
            # Afficher un exemple de document pour debug
            if documents:
                print("\nExemple de document:")
                print(json.dumps(documents[0], indent=2))
        
        return samples

    def _extract_chunks_with_metadata(self, doc: Dict) -> List[Dict]:
        chunks_with_meta = []
        
        try:
            if 'chunks' not in doc:
                print(f"Warning: Document sans chunks: {doc.get('title', 'Unknown')}")
                return chunks_with_meta
                
            for chunk in doc['chunks']:
                if not chunk.get('text', '').strip():
                    continue
                    
                chunks_with_meta.append({
                    'text': chunk['text'],
                    'hierarchy': [doc['title'], chunk.get('section', 'Unknown Section')],
                    'position': chunk['position']
                })
                
            if not chunks_with_meta:
                print(f"Warning: Aucun chunk valide extrait du document: {doc.get('title', 'Unknown')}")
                
        except Exception as e:
            print(f"Erreur lors de l'extraction des chunks: {str(e)}")
            print(f"Document problématique: {doc}")
            
        return chunks_with_meta

    def _create_input_sequence(self, sequence: List[Dict]) -> torch.Tensor:
        """Crée la séquence d'embeddings d'entrée."""
        embeddings = []
        
        if not sequence:
            raise ValueError("Séquence vide")
            
        for chunk in sequence:
            if not chunk['text'].strip():
                continue
                
            # Embedding du texte
            text_emb = self.embedding_model.encode(chunk['text'])
            embeddings.append(text_emb)
            
        if not embeddings:
            raise ValueError("Aucun embedding valide créé")
            
        return torch.tensor(np.stack(embeddings))
    
    def _create_target_embedding(self, 
                               center_chunk: Dict, 
                               context_sequence: List[Dict]) -> torch.Tensor:
        """Crée l'embedding cible enrichi pour le chunk central."""
        # Embedding de base du chunk central
        base_embedding = self.embedding_model.encode(center_chunk['text'])
        
        # Enrichissement avec informations structurelles
        hierarchy_text = " > ".join(center_chunk['hierarchy'])
        hierarchy_embedding = self.embedding_model.encode(hierarchy_text)
        
        # Enrichissement avec le contexte local
        context_text = " ".join([c['text'] for c in context_sequence])
        context_embedding = self.embedding_model.encode(context_text)
        
        # Combiner les embeddings (exemple simple avec moyenne pondérée)
        target = (0.5 * base_embedding + 
                 0.25 * hierarchy_embedding + 
                 0.25 * context_embedding)
        
        return torch.tensor(target)
    
    def _split_into_chunks(self, text: str, chunk_size: int = 3) -> List[str]:
        sentences = nltk.sent_tokenize(text)
        chunks = []
        
        for i in range(0, len(sentences), chunk_size):
            chunk = ' '.join(sentences[i:i + chunk_size])
            if chunk.strip():  # Vérifier que le chunk n'est pas vide
                chunks.append(chunk)
                
        return chunks
    
    def __len__(self):
        """Retourne le nombre total d'échantillons."""
        return len(self.samples)
    
    # Ajouter cette méthode aussi
    def __getitem__(self, idx):
        """Retourne un échantillon spécifique."""
        return self.samples[idx]
    

In [69]:
# Fonction de collate personnalisée
def collate_sequences(batch):
    """
    Prépare un batch de données avec padding et masques d'attention.
    """
    # Trier par longueur pour un padding efficace
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    
    # Obtenir la longueur maximale dans ce batch
    max_len = len(batch[0][0])
    
    padded_inputs = []
    padded_targets = []
    attention_masks = []
    
    for input_seq, target in batch:
        seq_len = len(input_seq)
        padding_len = max_len - seq_len
        
        # Padding des inputs
        padded_seq = torch.cat([
            input_seq,
            torch.zeros((padding_len, input_seq.size(1)), device=input_seq.device)
        ])
        padded_inputs.append(padded_seq)
        
        # Masque d'attention (1 pour les vrais tokens, 0 pour le padding)
        mask = torch.cat([
            torch.ones(seq_len, device=input_seq.device),
            torch.zeros(padding_len, device=input_seq.device)
        ])
        attention_masks.append(mask)
        
        # Targets (pas besoin de padding car c'est un seul embedding)
        padded_targets.append(target)
    
    return {
        'input_sequences': torch.stack(padded_inputs),
        'attention_masks': torch.stack(attention_masks),
        'targets': torch.stack(padded_targets)
    }

## Validation and Testing

### Data Validation

In [70]:
# Classe d'évaluation
class EmbeddingEvaluator:
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        
    def evaluate_embeddings(self, 
                          original_emb: torch.Tensor, 
                          enriched_emb: torch.Tensor, 
                          context_emb: torch.Tensor,
                          hierarchy_info: Dict = None) -> Dict[str, float]:
        """
        Évalue la qualité des embeddings enrichis selon plusieurs métriques.
        """
        # Convertir en numpy pour les calculs de similarité
        original_np = original_emb.detach().cpu().numpy()
        enriched_np = enriched_emb.detach().cpu().numpy()
        context_np = context_emb.detach().cpu().numpy()
        
        # Reshape pour cosine_similarity si nécessaire
        if len(original_np.shape) == 1:
            original_np = original_np.reshape(1, -1)
            enriched_np = enriched_np.reshape(1, -1)
            context_np = context_np.reshape(1, -1)
        
        # Calculer les similarités
        original_sim = float(cosine_similarity(enriched_np, original_np)[0, 0])
        context_sim = float(cosine_similarity(enriched_np, context_np)[0, 0])
        
        # Évaluer la préservation de la hiérarchie si l'information est fournie
        hierarchy_score = self.evaluate_hierarchy_preservation(
            enriched_np, hierarchy_info) if hierarchy_info else 0.0
        
        return {
            'original_similarity': original_sim,
            'context_similarity': context_sim,
            'hierarchy_score': hierarchy_score,
            'combined_score': (original_sim + context_sim + hierarchy_score) / 3
        }
    
    def evaluate_hierarchy_preservation(self, 
                                     embedding: np.ndarray, 
                                     hierarchy_info: Dict) -> float:
        """
        Évalue si l'embedding préserve bien les relations hiérarchiques.
        """
        if not hierarchy_info:
            return 0.0
        
        # Exemple simple: comparer avec les embeddings des niveaux hiérarchiques
        hierarchy_embs = [
            self.embedding_model.encode(level) 
            for level in hierarchy_info['hierarchy']
        ]
        
        # Calculer la moyenne des similarités avec chaque niveau
        similarities = [
            float(cosine_similarity(
                embedding.reshape(1, -1), 
                h_emb.reshape(1, -1)
            )[0, 0])
            for h_emb in hierarchy_embs
        ]
        
        # Pondérer plus fortement les niveaux hiérarchiques proches
        weights = np.linspace(1.0, 0.5, len(similarities))
        weighted_avg = np.average(similarities, weights=weights)
        
        return weighted_avg

In [71]:
def create_dataloader(
    documents: List[Dict],
    embedding_model,
    batch_size: int = 32,
    window_size: int = 5,
    max_seq_length: int = 50,
    num_workers: int = 0,  # Changé à 0 pour le debug
    use_augmentation: bool = True
) -> DataLoader:
    print(f"Création du DataLoader avec {len(documents)} documents")
    
    dataset = ContextualEmbeddingDataset(
        documents=documents,
        embedding_model=embedding_model,
        window_size=window_size,
        max_seq_length=max_seq_length,
        use_augmentation=use_augmentation
    )
    
    print(f"Dataset créé avec {len(dataset)} échantillons")
    if len(dataset) == 0:
        raise ValueError("Le dataset est vide!")
    
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        collate_fn=collate_sequences
    )

In [83]:
def train_epoch(
    model, 
    dataloader, 
    optimizer, 
    evaluator, 
    device='cuda',
    gradient_clip: float = None
):
    """
    Trains the model for one epoch.
    
    Args:
        model: The model to train
        dataloader: DataLoader with training data
        optimizer: Optimizer instance
        evaluator: Evaluator instance
        device: Device to use (default: 'cuda')
        gradient_clip: Maximum gradient norm (optional)
        
    Returns:
        dict: Training metrics for the epoch
    """
    model.train()
    total_loss = 0
    all_metrics = []
    
    for batch in dataloader:
        # Move data to device
        input_sequences = batch['input_sequences'].to(device)
        attention_masks = batch['attention_masks'].to(device)
        targets = batch['targets'].to(device)
        
        # Forward pass
        enriched = model(input_sequences, attention_masks)
        
        # Calculate loss
        loss = torch.nn.functional.mse_loss(enriched, targets)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping if specified
        if gradient_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            
        optimizer.step()
        
        # Evaluate batch
        with torch.no_grad():
            batch_metrics = {}
            for i in range(len(input_sequences)):
                metrics = evaluator.evaluate_embeddings(
                    input_sequences[i],
                    enriched[i],
                    targets[i]
                )
                for k, v in metrics.items():
                    batch_metrics[k] = batch_metrics.get(k, 0) + v
                    
            # Average batch metrics
            batch_metrics = {k: v / len(input_sequences) for k, v in batch_metrics.items()}
            batch_metrics['loss'] = loss.item()
            all_metrics.append(batch_metrics)
        
        total_loss += loss.item()
    
    # Calculate average metrics across all batches
    avg_metrics = {}
    for metric in all_metrics[0].keys():
        avg_metrics[metric] = sum(batch[metric] for batch in all_metrics) / len(all_metrics)
        
    return avg_metrics

def validate_epoch(model, dataloader, evaluator, device='cuda'):
    """
    Validates the model for one epoch.
    
    Args:
        model: The model to validate
        dataloader: DataLoader with validation data
        evaluator: Evaluator instance
        device: Device to use (default: 'cuda')
        
    Returns:
        dict: Validation metrics for the epoch
    """
    model.eval()
    total_loss = 0
    all_metrics = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_sequences = batch['input_sequences'].to(device)
            attention_masks = batch['attention_masks'].to(device)
            targets = batch['targets'].to(device)
            
            # Forward pass
            enriched = model(input_sequences, attention_masks)
            
            # Calculate loss
            loss = torch.nn.functional.mse_loss(enriched, targets)
            
            # Evaluate batch
            batch_metrics = {}
            for i in range(len(input_sequences)):
                metrics = evaluator.evaluate_embeddings(
                    input_sequences[i],
                    enriched[i],
                    targets[i]
                )
                for k, v in metrics.items():
                    batch_metrics[k] = batch_metrics.get(k, 0) + v
                    
            # Average batch metrics
            batch_metrics = {k: v / len(input_sequences) for k, v in batch_metrics.items()}
            batch_metrics['loss'] = loss.item()
            all_metrics.append(batch_metrics)
            
            total_loss += loss.item()
    
    # Calculate average metrics across all batches
    avg_metrics = {}
    for metric in all_metrics[0].keys():
        avg_metrics[metric] = sum(batch[metric] for batch in all_metrics) / len(all_metrics)
        
    return avg_metrics

def log_metrics(writer, train_metrics: dict, val_metrics: dict, epoch: int):
    """
    Logs metrics to tensorboard.
    
    Args:
        writer: SummaryWriter instance
        train_metrics: Training metrics
        val_metrics: Validation metrics
        epoch: Current epoch number
    """
    for metric_name in train_metrics:
        writer.add_scalars(
            f'metrics/{metric_name}',
            {
                'train': train_metrics[metric_name],
                'val': val_metrics.get(metric_name, 0)
            },
            epoch
        )

def save_checkpoint(model, optimizer, epoch, metrics, config, path):
    """
    Saves a checkpoint of the model.
    
    Args:
        model: Model to save
        optimizer: Optimizer state
        epoch: Current epoch
        metrics: Current metrics
        config: Model configuration
        path: Path to save the checkpoint
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
        'config': config
    }
    torch.save(checkpoint, path)

## Embedding Models

This section defines the embedding models used for text encoding. We support multiple embedding models including:
- Sentence Transformers (default)
- Hugging Face transformers
- Custom trained models

In [84]:
class EmbeddingUtils:
    @staticmethod
    def calculate_similarity(emb1: np.ndarray, emb2: np.ndarray) -> float:
        """Calcule la similarité cosinus entre deux embeddings."""
        return float(cosine_similarity(emb1.reshape(1, -1), emb2.reshape(1, -1))[0, 0])
    
    @staticmethod
    def aggregate_embeddings(embeddings: List[np.ndarray], 
                           weights: List[float] = None) -> np.ndarray:
        """Agrège plusieurs embeddings avec des poids optionnels."""
        if weights is None:
            weights = [1.0] * len(embeddings)
        
        weighted_sum = sum(w * e for w, e in zip(weights, embeddings))
        return weighted_sum / sum(weights)
    
    @staticmethod
    def visualize_embeddings(embeddings: np.ndarray, 
                           labels: List[str] = None,
                           method: str = 'tsne'):
        """Visualise les embeddings en 2D."""
        if method == 'tsne':
            from sklearn.manifold import TSNE
            reducer = TSNE(n_components=2, random_state=42)
        else:  # 'pca'
            from sklearn.decomposition import PCA
            reducer = PCA(n_components=2)
            
        embedded = reducer.fit_transform(embeddings)
        
        plt.figure(figsize=(10, 8))
        plt.scatter(embedded[:, 0], embedded[:, 1])
        
        if labels:
            for i, label in enumerate(labels):
                plt.annotate(label, (embedded[i, 0], embedded[i, 1]))
                
        plt.title(f'Embeddings visualization using {method.upper()}')
        plt.show()

In [85]:


class EmbeddingModel:
    def __init__(
        self,
        model_name: str = "paraphrase-multilingual-mpnet-base-v2",
        model_type: str = "sentence_transformers",
        device: str = None,
        max_length: int = 512
    ):
        """
        Initialise le modèle d'embedding.
        
        Args:
            model_name: Nom du modèle à utiliser
            model_type: Type de modèle ('sentence_transformers' ou 'huggingface')
            device: Device à utiliser ('cuda' ou 'cpu')
            max_length: Longueur maximale des séquences
        """
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.max_length = max_length
        self.model_type = model_type
        
        if model_type == "sentence_transformers":
            self.model = SentenceTransformer(model_name).to(self.device)
        elif model_type == "huggingface":
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name).to(self.device)
        else:
            raise ValueError(f"Model type {model_type} not supported")
            
    def mean_pooling(self, model_output, attention_mask):
        """Moyenne des tokens pour obtenir l'embedding de phrase."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    @torch.no_grad()
    def encode(self, texts: Union[str, List[str]], batch_size: int = 32) -> np.ndarray:
        """
        Encode les textes en embeddings.
        
        Args:
            texts: Texte ou liste de textes à encoder
            batch_size: Taille du batch pour l'encodage
            
        Returns:
            np.ndarray: Embeddings des textes
        """
        # Normaliser l'entrée
        if isinstance(texts, str):
            texts = [texts]
            
        # Encoder selon le type de modèle
        if self.model_type == "sentence_transformers":
            embeddings = self.model.encode(
                texts,
                batch_size=batch_size,
                show_progress_bar=False,
                convert_to_numpy=True
            )
        else:  # huggingface
            embeddings = []
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i + batch_size]
                encoded = self.tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors='pt'
                ).to(self.device)
                
                model_output = self.model(**encoded)
                batch_embeddings = self.mean_pooling(model_output, encoded['attention_mask'])
                batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
                embeddings.append(batch_embeddings.cpu().numpy())
                
            embeddings = np.vstack(embeddings)
            
        return embeddings
    
    def get_embedding_dim(self) -> int:
        """Retourne la dimension des embeddings."""
        if self.model_type == "sentence_transformers":
            return self.model.get_sentence_embedding_dimension()
        else:
            return self.model.config.hidden_size
            
    def __call__(self, texts: Union[str, List[str]], **kwargs) -> np.ndarray:
        """Permet d'utiliser l'instance comme une fonction."""
        return self.encode(texts, **kwargs)

In [86]:
class EmbeddingModelFactory:
    @staticmethod
    def create_model(
        model_config: Dict[str, Any]
    ) -> EmbeddingModel:
        """
        Crée une instance de modèle d'embedding selon la configuration.
        
        Args:
            model_config: Configuration du modèle
            
        Returns:
            EmbeddingModel: Instance du modèle d'embedding
        """
        default_config = {
            "model_name": "paraphrase-multilingual-mpnet-base-v2",
            "model_type": "sentence_transformers",
            "device": None,
            "max_length": 512
        }
        
        # Mise à jour de la configuration par défaut
        config = {**default_config, **model_config}
        
        return EmbeddingModel(**config)

In [87]:
def split_documents(
    documents: List[Dict],
    train_size: float = 0.7,
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """
    Sépare les documents en ensembles d'entraînement, validation et test.
    
    Args:
        documents: Liste des documents à séparer
        train_size: Proportion pour l'entraînement
        val_size: Proportion pour la validation
        test_size: Proportion pour le test
        random_state: Seed pour la reproductibilité
        
    Returns:
        Tuple contenant les documents train, val et test
    """
    if abs(train_size + val_size + test_size - 1.0) > 1e-6:
        raise ValueError("Les proportions doivent sommer à 1")
        
    # Première séparation pour isoler le test
    train_val, test = train_test_split(
        documents,
        test_size=test_size,
        random_state=random_state
    )
    
    # Seconde séparation pour séparer train et validation
    relative_val_size = val_size / (train_size + val_size)
    train, val = train_test_split(
        train_val,
        test_size=relative_val_size,
        random_state=random_state
    )
    
    print(f"Split sizes: Train={len(train)}, Val={len(val)}, Test={len(test)}")
    return train, val, test

class EarlyStopping:
    def __init__(
        self,
        patience: int = 3,
        min_delta: float = 1e-4,
        mode: str = 'min'
    ):
        """
        Gestionnaire d'early stopping.
        
        Args:
            patience: Nombre d'époques à attendre avant l'arrêt
            min_delta: Changement minimum considéré comme une amélioration
            mode: 'min' pour minimisation, 'max' pour maximisation
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_value = float('inf') if mode == 'min' else float('-inf')
        self.early_stop = False
        
    def __call__(self, current_value: float) -> bool:
        """
        Vérifie si l'entraînement doit être arrêté.
        
        Args:
            current_value: Valeur courante de la métrique
            
        Returns:
            bool: True si l'entraînement doit être arrêté
        """
        if self.mode == 'min':
            improvement = self.best_value - current_value > self.min_delta
        else:
            improvement = current_value - self.best_value > self.min_delta
            
        if improvement:
            self.best_value = current_value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                
        return self.early_stop

def should_stop_early(
    current_loss: float,
    patience: int = 3,
    min_delta: float = 1e-4
) -> bool:
    """
    Wrapper pour EarlyStopping.
    
    Args:
        current_loss: Perte courante
        patience: Nombre d'époques à attendre
        min_delta: Changement minimum pour une amélioration
        
    Returns:
        bool: True si l'entraînement doit être arrêté
    """
    if not hasattr(should_stop_early, 'early_stopping'):
        should_stop_early.early_stopping = EarlyStopping(
            patience=patience,
            min_delta=min_delta
        )
    
    return should_stop_early.early_stopping(current_loss)

def load_checkpoint(
    path: str,
    model: HybridEnricher = None,
    device: str = 'cuda'
) -> Tuple[HybridEnricher, Dict]:
    """
    Charge un modèle sauvegardé.
    
    Args:
        path: Chemin vers le checkpoint
        model: Instance de modèle (optionnel)
        device: Device à utiliser
        
    Returns:
        Tuple contenant le modèle chargé et les informations du checkpoint
    """
    checkpoint = torch.load(path, map_location=device)
    
    if model is None:
        config = checkpoint['config']
        model = HybridEnricher(
            embedding_dim=config['embedding_dim'],
            hidden_dim=config['hidden_dim'],
            nhead=config['model']['num_heads'],
            num_layers=config['model']['num_layers'],
            dropout=config['model']['dropout']
        ).to(device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    
    return model, checkpoint

def evaluate_model(
    model: HybridEnricher,
    test_documents: List[Dict],
    embedding_model,
    evaluator: EmbeddingEvaluator,
    batch_size: int = 32,
    device: str = 'cuda'
) -> Dict[str, float]:
    """
    Évalue le modèle sur un ensemble de test.
    
    Args:
        model: Modèle à évaluer
        test_documents: Documents de test
        embedding_model: Modèle d'embedding
        evaluator: Évaluateur
        batch_size: Taille des batches
        device: Device à utiliser
        
    Returns:
        Dict contenant les métriques d'évaluation
    """
    model.eval()
    test_loader = create_dataloader(
        documents=test_documents,
        embedding_model=embedding_model,
        batch_size=batch_size,
        use_augmentation=False
    )
    
    metrics_sum = defaultdict(float)
    total_samples = 0
    
    with torch.no_grad():
        for batch in test_loader:
            input_sequences = batch['input_sequences'].to(device)
            attention_masks = batch['attention_masks'].to(device)
            targets = batch['targets'].to(device)
            
            # Forward pass
            enriched = model(input_sequences, attention_masks)
            
            # Évaluation batch par batch
            for i in range(len(input_sequences)):
                metrics = evaluator.evaluate_embeddings(
                    input_sequences[i],
                    enriched[i],
                    targets[i]
                )
                
                for metric_name, value in metrics.items():
                    metrics_sum[metric_name] += value
                    
            total_samples += len(input_sequences)
    
    # Calcul des moyennes
    avg_metrics = {
        name: value / total_samples 
        for name, value in metrics_sum.items()
    }
    
    # Ajouter des métriques supplémentaires si nécessaire
    avg_metrics.update({
        'num_samples': total_samples,
        'num_documents': len(test_documents)
    })
    
    return avg_metrics

# Fonction utilitaire pour l'analyse détaillée des erreurs
def analyze_errors(
    model: HybridEnricher,
    test_documents: List[Dict],
    embedding_model,
    evaluator: EmbeddingEvaluator,
    n_worst: int = 10
) -> Dict:
    """
    Analyse détaillée des erreurs du modèle.
    
    Args:
        model: Modèle à analyser
        test_documents: Documents de test
        embedding_model: Modèle d'embedding
        evaluator: Évaluateur
        n_worst: Nombre de pires cas à retourner
        
    Returns:
        Dict contenant l'analyse des erreurs
    """
    model.eval()
    error_cases = []
    
    with torch.no_grad():
        for doc in test_documents:
            chunks = doc['chunks']  # Supposons que c'est la structure
            
            for i, chunk in enumerate(chunks):
                # Créer le contexte
                context = chunks[max(0, i-2):i] + chunks[i+1:i+3]
                
                # Obtenir l'embedding enrichi
                enriched = process_sequence(
                    chunks=[chunk] + context,
                    embedding_model=embedding_model,
                    enricher=model
                )[0]  # Premier élément car on ne veut que l'embedding du chunk central
                
                # Évaluer l'erreur
                metrics = evaluator.evaluate_embeddings(
                    original_emb=embedding_model.encode(chunk['text']),
                    enriched_emb=enriched,
                    context_emb=embedding_model.encode(' '.join([c['text'] for c in context]))
                )
                
                error_cases.append({
                    'chunk': chunk,
                    'context': context,
                    'metrics': metrics,
                    'error': 1 - metrics['combined_score']  # Plus c'est haut, pire c'est
                })
    
    # Trier par erreur décroissante
    error_cases.sort(key=lambda x: x['error'], reverse=True)
    
    return {
        'worst_cases': error_cases[:n_worst],
        'average_error': sum(case['error'] for case in error_cases) / len(error_cases),
        'error_std': np.std([case['error'] for case in error_cases]),
        'total_analyzed': len(error_cases)
    }

In [88]:
def download_and_prepare_data(
    num_articles: int = 1000,
    lang: str = 'fr',
    cache_dir: str = './data'
) -> List[Dict]:
    """
    Télécharge et prépare les données Wikipedia.
    """
    from datasets import load_dataset
    import nltk
    from tqdm import tqdm
    import os
    import json
    
    print("Configuration de NLTK...")
    nltk.download('punkt')
    os.makedirs(cache_dir, exist_ok=True)
    
    print(f"Chargement du dataset Wikipedia ({lang})...")
    wiki_dataset = load_dataset(
        'wikipedia',
        f'20220301.{lang}',
        cache_dir=cache_dir,
        trust_remote_code=True
    )
    
    # Prendre un sous-ensemble des articles
    articles = wiki_dataset['train'].select(range(num_articles))
    
    print("Traitement des articles...")
    processed_documents = []
    for article in tqdm(articles):
        processed_doc = process_article(article)
        if processed_doc and processed_doc['chunks']:
            processed_documents.append(processed_doc)
    
    print(f"Articles traités avec succès: {len(processed_documents)}/{num_articles}")
    
    # Sauvegarder un exemple de document pour debug
    if processed_documents:
        debug_path = os.path.join(cache_dir, 'example_processed_doc.json')
        with open(debug_path, 'w', encoding='utf-8') as f:
            json.dump(processed_documents[0], f, indent=4, ensure_ascii=False)
            
    if not processed_documents:
        raise ValueError("Aucun document valide n'a été traité!")
        
    return processed_documents
    
def process_article(article) -> Dict:
    """Traite un article Wikipedia."""
    try:
        # Extraire le texte et le titre
        text = article['text']
        title = article['title']
        
        if not text.strip() or not title.strip():
            print(f"Article ignoré: texte ou titre vide")
            return None
            
        # Découper en sections (simplification basique)
        sections = text.split('\n\n')
        
        # Découper chaque section en chunks
        chunks = []
        current_section = "Introduction"  # Section par défaut
        
        for section in sections:
            if not section.strip():
                continue
                
            # Si la ligne commence par '=' c'est un titre de section
            if section.strip().startswith('='):
                current_section = section.strip().replace('=', '').strip()
                continue
                
            # Découper la section en phrases
            sentences = nltk.sent_tokenize(section)
            
            if not sentences:
                continue
                
            # Créer des chunks de 3 phrases avec chevauchement
            for i in range(0, len(sentences), 2):
                chunk_sentences = sentences[i:i+3]
                if chunk_sentences:
                    chunk_text = ' '.join(chunk_sentences)
                    if len(chunk_text.split()) >= 5:  # Ignorer les chunks trop courts
                        chunks.append({
                            'text': chunk_text,
                            'section': current_section,
                            'position': len(chunks),
                            'article_title': title
                        })
        
        if not chunks:
            print(f"Article ignoré: aucun chunk valide - {title}")
            return None
            
        return {
            'title': title,
            'chunks': chunks,
            'metadata': {
                'url': article.get('url', ''),
                'num_chunks': len(chunks)
            }
        }
    except Exception as e:
        print(f"Erreur lors du traitement de l'article {article.get('title', 'Unknown')}: {str(e)}")
        return None

In [89]:
# Modification de la fonction main pour inclure le chargement des données
def main():
    """
    Fonction principale pour l'entraînement et l'évaluation du modèle d'enrichissement contextuel.
    """
    
    # Configuration
    config = {
        'embedding': {
            'model_name': 'paraphrase-multilingual-mpnet-base-v2',
            'model_type': 'sentence_transformers',
            'max_length': 512
        },
        'training': {
            'num_epochs': 10,
            'batch_size': 32,
            'learning_rate': 1e-4,
            'weight_decay': 1e-5,
            'gradient_clip': 1.0,
            'train_size': 0.7,
            'val_size': 0.15,
            'test_size': 0.15
        },
        'model': {
            'num_heads': 8,
            'num_layers': 2,
            'dropout': 0.1
        },
        'data': {
            'window_size': 5,
            'max_seq_length': 50,
            'use_augmentation': True,
            'num_articles': 10,
            'language': 'fr',
            'cache_dir': './data'
        },
        'early_stopping': {
            'patience': 3,
            'min_delta': 1e-4
        }
    }

    # Initialisation du logging

    run_name = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = os.path.join("runs", run_name)
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir)
    
    # Sauvegarde de la configuration
    with open(os.path.join(log_dir, 'config.json'), 'w') as f:
        json.dump(config, f, indent=4)
    
    print("Téléchargement et préparation des données...")
    documents = download_and_prepare_data(
        num_articles=config['data']['num_articles'],
        lang=config['data']['language'],
        cache_dir=config['data']['cache_dir']
    )
    
    print(f"Nombre total de documents: {len(documents)}")
    
    # Sauvegarde d'un exemple de document pour référence
    with open(os.path.join(log_dir, 'example_document.json'), 'w') as f:
        json.dump(documents[0], f, indent=4)
    
    print("Initialisation du modèle d'embedding...")
    embedding_model = EmbeddingModelFactory.create_model(config['embedding'])
    embedding_dim = embedding_model.get_embedding_dim()
    hidden_dim = embedding_dim // 2

    print("Initialisation du modèle principal...")
    model = HybridEnricher(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        nhead=config['model']['num_heads'],
        num_layers=config['model']['num_layers'],
        dropout=config['model']['dropout']
    ).to('cuda')
    
    print("Initialisation de l'évaluateur...")
    evaluator = EmbeddingEvaluator(embedding_model)

    print("Préparation des données...")
    train_documents, val_documents, test_documents = split_documents(
        documents=documents,
        train_size=config['training']['train_size'],
        val_size=config['training']['val_size'],
        test_size=config['training']['test_size']
    )
    
    train_loader = create_dataloader(
        documents=train_documents,
        embedding_model=embedding_model,
        batch_size=config['training']['batch_size'],
        window_size=config['data']['window_size'],
        max_seq_length=config['data']['max_seq_length'],
        use_augmentation=config['data']['use_augmentation']
    )
    
    val_loader = create_dataloader(
        documents=val_documents,
        embedding_model=embedding_model,
        batch_size=config['training']['batch_size'],
        window_size=config['data']['window_size'],
        max_seq_length=config['data']['max_seq_length'],
        use_augmentation=False
    )

    print("Configuration de l'optimiseur...")
    # Optimiseur et scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['training']['learning_rate'],
        weight_decay=config['training']['weight_decay']
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2,
        verbose=True
    )

    # Early stopping
    early_stopping = EarlyStopping(
        patience=config['early_stopping']['patience'],
        min_delta=config['early_stopping']['min_delta']
    )

    # Meilleurs métriques pour le checkpointing
    best_val_loss = float('inf')
    best_model_path = os.path.join(log_dir, 'best_model.pth')

    print("Début de l'entraînement...")
    val_metrics = None
    try:
        # Boucle d'entraînement
        for epoch in range(config['training']['num_epochs']):
            print(f"\nEpoch {epoch+1}/{config['training']['num_epochs']}")
            
            # Entraînement
            train_metrics = train_epoch(
                model=model,
                dataloader=train_loader,
                optimizer=optimizer,
                evaluator=evaluator,
                gradient_clip=config['training']['gradient_clip']
            )
            
            # Validation
            val_metrics = validate_epoch(
                model=model,
                dataloader=val_loader,
                evaluator=evaluator
            )
            
            # Logging
            log_metrics(writer, train_metrics, val_metrics, epoch)
            
            # Affichage des métriques
            print(f"Train Loss: {train_metrics['loss']:.4f}")
            print(f"Val Loss: {val_metrics['loss']:.4f}")
            
            # Scheduler step
            scheduler.step(val_metrics['loss'])
            
            # Checkpointing
            if val_metrics['loss'] < best_val_loss:
                best_val_loss = val_metrics['loss']
                save_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    epoch=epoch,
                    metrics=val_metrics,
                    config=config,
                    path=best_model_path
                )
                print(f"Nouveau meilleur modèle sauvegardé (loss: {best_val_loss:.4f})")
                
            # Early stopping check
            if early_stopping(val_metrics['loss']):
                print("Early stopping triggered")
                break
                
    except KeyboardInterrupt:
        print("\nEntraînement interrompu par l'utilisateur")
    
    finally:
        print("\nSauvegarde finale du modèle...")
        # Sauvegarde finale
        final_model_path = os.path.join(log_dir, 'final_model.pth')
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            metrics=val_metrics,
            config=config,
            path=final_model_path
        )
        
        # Fermeture du writer
        writer.close()
    
    print("\nChargement du meilleur modèle pour l'évaluation finale...")
    # Test final sur les meilleures métriques
    best_model, _ = load_checkpoint(best_model_path)
    
    print("Évaluation sur l'ensemble de test...")
    test_metrics = evaluate_model(
        model=best_model,
        test_documents=test_documents,
        embedding_model=embedding_model,
        evaluator=evaluator
    )
    
    print("\nAnalyse des erreurs...")
    error_analysis = analyze_errors(
        model=best_model,
        test_documents=test_documents,
        embedding_model=embedding_model,
        evaluator=evaluator,
        n_worst=10
    )
    
    # Sauvegarde des résultats finaux
    results = {
        'test_metrics': test_metrics,
        'error_analysis': error_analysis
    }
    
    with open(os.path.join(log_dir, 'final_results.json'), 'w') as f:
        json.dump(results, f, indent=4)
    
    print("\nRésultats finaux:")
    print("Test Metrics:")
    for metric_name, value in test_metrics.items():
        print(f"{metric_name}: {value:.4f}")
        
    print("\nAnalyse des erreurs:")
    print(f"Erreur moyenne: {error_analysis['average_error']:.4f}")
    print(f"Écart-type des erreurs: {error_analysis['error_std']:.4f}")
    print(f"\nNombre total d'exemples analysés: {error_analysis['total_analyzed']}")
    
    print("\nPires cas:")
    for i, case in enumerate(error_analysis['worst_cases']):
        print(f"\nCas {i+1}:")
        print(f"Erreur: {case['error']:.4f}")
        print(f"Texte: {case['chunk']['text'][:100]}...")
        
    print(f"\nEntraînement terminé. Tous les résultats sont sauvegardés dans: {log_dir}")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"Une erreur est survenue: {str(e)}")
        raise

Téléchargement et préparation des données...
Configuration de NLTK...
Chargement du dataset Wikipedia (fr)...


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\fleut\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Traitement des articles...


100%|██████████| 10/10 [00:00<00:00, 486.75it/s]

Articles traités avec succès: 10/10
Nombre total de documents: 10
Initialisation du modèle d'embedding...


Initialisation du modèle principal...
Initialisation de l'évaluateur...
Préparation des données...
Split sizes: Train=6, Val=2, Test=2
Création du DataLoader avec 6 documents
Initialisation du dataset avec 6 documents
Préparation des échantillons pour 6 documents
Document 0: 10 chunks extraits
Document 1: 97 chunks extraits
Document 2: 122 chunks extraits
Document 3: 39 chunks extraits
Document 4: 12 chunks extraits
Document 5: 58 chunks extraits
Statistiques de préparation des données:
- Documents traités: 6
- Total chunks extraits: 338
- Échantillons créés: 1014
Dataset initialisé avec 1014 échantillons
Dataset créé avec 1014 échantillons
Création du DataLoader avec 2 documents
Initialisation du dataset avec 2 documents
Préparation des échantillons pour 2 documents
Document 0: 32 chunks extraits
Document 1: 33 chunks extraits
Statistiques de préparation des données:
- Documents traités: 2
- Total chunks extraits: 65
- Échantillons créés: 65
Dataset initialisé avec 65 échantillons
Dat

RuntimeError: Tensors must have same number of dimensions: got 3 and 2